In [1]:
import pandas as pd

df = pd.read_csv("./data/test.csv")

In [2]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3513 entries, 0 to 3512
Data columns (total 16 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   ChemID               3513 non-null   int64  
 1   source_casrn         3381 non-null   object 
 2   CAS                  3513 non-null   object 
 3   SMILES               3513 non-null   object 
 4   NAME                 3513 non-null   object 
 5   preferred_name       1929 non-null   object 
 6   iupac                1903 non-null   object 
 7   MPID                 3513 non-null   object 
 8   dsstox_substance_id  3383 non-null   object 
 9   Canonical_QSARr      3513 non-null   object 
 10  InChI_Code_QSARr     3513 non-null   object 
 11  InChI Key_QSARr      3513 non-null   object 
 12  Salt_Solvent         3510 non-null   object 
 13  Salt_Solvent_ID      26 non-null     float64
 14  Kow Reference        3510 non-null   object 
 15  LogP                 3513 non-null   f

In [3]:
df = df.dropna(axis=1, how='any') # 결측치 다 제거거

In [4]:
import pandas as pd
from rdkit import Chem
from mordred import Calculator, descriptors

# -----------------------------------------
# 1) Mordred 전체 descriptor calculator 생성
# -----------------------------------------
calc = Calculator(descriptors, ignore_3D=True)
# ignore_3D=True → 3D 좌표 필요 없는 descriptor만 계산 (SMILES 기반이므로 필수)

# -----------------------------------------
# 2) SMILES → RDKit Mol 변환 (에러 처리 포함)
# -----------------------------------------
def smiles_to_mol(smiles):
    try:
        mol = Chem.MolFromSmiles(smiles)
    except:
        mol = None
    return mol

df["mol"] = df["SMILES"].apply(smiles_to_mol)

# ------------------------------
# 3) Descriptor 계산 (시간 오래 걸림)
# ------------------------------
# drop invalid molecules
valid_df = df[df["mol"].notnull()].copy()

# 실제 계산
result_df = calc.pandas(valid_df["mol"])

result_df.to_csv("sample.csv", index=False)
# --------------------------------------
# 1) result_df에서 mol 제거 (혹시 남아 있다면)
# --------------------------------------
if "mol" in result_df.columns:
    result_df = result_df.drop(columns=["mol"])

# --------------------------------------
# 2) df + result_df 가로로 합치기
#     (index가 동일하다는 전제)
# --------------------------------------
merged = pd.concat([df.reset_index(drop=True), 
                    result_df.reset_index(drop=True)], axis=1)

print("🔍 합친 후 shape:", merged.shape)

# --------------------------------------
# 3) 결측치가 있는 row 전부 제거
# --------------------------------------
cleaned = merged.dropna(axis=0).reset_index(drop=True)

print("🧹 결측치 제거 후 shape:", cleaned.shape)

# --------------------------------------
# 4) LogP + descriptor만 남기기
#     → 원래 df의 column들은 제거
# --------------------------------------
# df의 original columns 리스트
original_cols = df.columns.tolist()

# LogP만 남기고 나머지 제거
cols_to_drop = [c for c in original_cols if c != "LogP"]

final_df = cleaned.drop(columns=cols_to_drop)

print("📌 최종 dataframe shape:", final_df.shape)
print("📌 남아 있는 column 수:", len(final_df.columns))

# --------------------------------------
# 5) 저장
# --------------------------------------
final_df.to_csv("final_mordred_dataset.csv", index=False)
print("💾 저장 완료: final_mordred_dataset.csv")


[14:39:43] Explicit valence for atom # 5 N, 4, is greater than permitted
 36%|███▌      | 1253/3512 [00:41<01:07, 33.58it/s]

/home/junsoo/miniconda3/envs/mordred/lib/python3.9/site-packages/numpy/_core/fromnumeric.py:86: RuntimeWarning: overflow encountered in reduce
  return ufunc.reduce(obj, axis, dtype, out, **passkwargs)


 57%|█████▋    | 1999/3512 [01:09<02:06, 11.93it/s]

/home/junsoo/miniconda3/envs/mordred/lib/python3.9/site-packages/numpy/_core/fromnumeric.py:86: RuntimeWarning: overflow encountered in reduce
  return ufunc.reduce(obj, axis, dtype, out, **passkwargs)


 74%|███████▍  | 2611/3512 [01:33<00:35, 25.28it/s]

/home/junsoo/miniconda3/envs/mordred/lib/python3.9/site-packages/numpy/_core/fromnumeric.py:86: RuntimeWarning: overflow encountered in reduce
  return ufunc.reduce(obj, axis, dtype, out, **passkwargs)


100%|██████████| 3512/3512 [02:11<00:00, 26.72it/s]


🔍 합친 후 shape: (3513, 1623)
🧹 결측치 제거 후 shape: (3511, 1623)
📌 최종 dataframe shape: (3511, 1614)
📌 남아 있는 column 수: 1614
💾 저장 완료: final_mordred_dataset.csv


In [8]:
final_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3511 entries, 0 to 3510
Columns: 1614 entries, LogP to mZagreb2
dtypes: float64(1078), object(536)
memory usage: 43.2+ MB


In [9]:
tmp = final_df.select_dtypes(exclude=['object'])


In [10]:
tmp.to_csv("test_dataset.csv", index=False)

In [9]:
tmp.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10517 entries, 0 to 10516
Columns: 1078 entries, LogP to mZagreb2
dtypes: float64(1078)
memory usage: 86.5 MB


In [ ]:
X = tmp.drop(columns='LogP')
y = tmp['LogP']

In [5]:

# --------------------------------------
# 1) result_df에서 mol 제거 (혹시 남아 있다면)
# --------------------------------------
if "mol" in result_df.columns:
    result_df = result_df.drop(columns=["mol"])

# -------------------------------
#-------
# 2) df + result_df 가로로 합치기
#     (index가 동일하다는 전제)
# --------------------------------------
merged = pd.concat([df.reset_index(drop=True), 
                    result_df.reset_index(drop=True)], axis=1)

print("🔍 합친 후 shape:", merged.shape)

# --------------------------------------
# 3) 결측치가 있는 row 전부 제거
# --------------------------------------
cleaned = merged.dropna(axis=0).reset_index(drop=True)

print("🧹 결측치 제거 후 shape:", cleaned.shape)

# --------------------------------------
# 4) LogP + descriptor만 남기기
#     → 원래 df의 column들은 제거
# --------------------------------------
# df의 original columns 리스트
original_cols = df.columns.tolist()

# LogP만 남기고 나머지 제거
cols_to_drop = [c for c in original_cols if c != "LogP"]

final_df = cleaned.drop(columns=cols_to_drop)

print("📌 최종 dataframe shape:", final_df.shape)
print("📌 남아 있는 column 수:", len(final_df.columns))

# --------------------------------------
# 5) 저장
# --------------------------------------
final_df.to_csv("final_mordred_dataset.csv", index=False)
print("💾 저장 완료: final_mordred_dataset.csv")


🔍 합친 후 shape: (10537, 1623)
🧹 결측치 제거 후 shape: (10517, 1623)
📌 최종 dataframe shape: (10517, 1614)
📌 남아 있는 column 수: 1614
💾 저장 완료: final_mordred_dataset.csv


In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10537 entries, 0 to 10536
Data columns (total 9 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   ChemID            10537 non-null  int64  
 1   CAS               10537 non-null  object 
 2   SMILES            10537 non-null  object 
 3   NAME              10537 non-null  object 
 4   MPID              10537 non-null  object 
 5   Canonical_QSARr   10537 non-null  object 
 6   InChI_Code_QSARr  10537 non-null  object 
 7   InChI Key_QSARr   10537 non-null  object 
 8   LogP              10537 non-null  float64
dtypes: float64(1), int64(1), object(7)
memory usage: 741.0+ KB


In [3]:
import pubchempy as pcp

from rdkit import Chem, DataStructs
from rdkit.Chem import AllChem
from rdkit.Chem.rdFingerprintGenerator import GetMorganGenerator

import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns



from mordred import Calculator, descriptors

from mordred.HydrogenBond import HBondDonor, HBondAcceptor
from mordred.RingCount import RingCount
from mordred.Weight import Weight
from mordred.TopoPSA import TopoPSA
from mordred.Aromatic import AromaticAtomsCount
from mordred.RotatableBond import RotatableBondsCount
from mordred.KappaShapeIndex import KappaShapeIndex1, KappaShapeIndex2, KappaShapeIndex3
from mordred.BalabanJ import BalabanJ
from mordred.ZagrebIndex import ZagrebIndex
from mordred.BertzCT import BertzCT

In [ ]:
important_desc = [
    HBondDonor(),
    HBondAcceptor(),
    RingCount(),
    Weight(),
    TopoPSA()
]

extra_desc = [
    AromaticAtomsCount(),
    RotatableBondsCount(),
    KappaShapeIndex1(),
    KappaShapeIndex2(),
    KappaShapeIndex3(),
    BalabanJ(),
    ZagrebIndex(),
    BertzCT(),
]

descriptor_list = important_desc + extra_desc

records = list()
calc = Calculator(descriptor_list, ignore_3D=True)

In [160]:
for idx, row in df.iterrows():
    smiles = row["SMILES"]

    mol = Chem.MolFromSmiles(smiles)
    
    if mol is None:
        print("there is no smiles matching")
        continue

    desc_values = calc(mol).asdict()
    desc_values["Name"] = row["Name"]
    desc_values["SMILES"] = smiles

    records.append(desc_values)

    if idx % 500 == 0:
        print(f"[INFO] Processed {idx} / {len(df)} molecules")


desc_df = pd.DataFrame(records)

[INFO] Processed 0 / 10534 molecules
[INFO] Processed 500 / 10534 molecules
[INFO] Processed 1000 / 10534 molecules


[00:07:32] Explicit valence for atom # 7 N, 4, is greater than permitted


there is no smiles matching
[INFO] Processed 1500 / 10534 molecules
[INFO] Processed 2000 / 10534 molecules


[00:07:34] Explicit valence for atom # 7 N, 4, is greater than permitted


there is no smiles matching
[INFO] Processed 2500 / 10534 molecules
[INFO] Processed 3000 / 10534 molecules
[INFO] Processed 3500 / 10534 molecules


...

there is no smiles matching
there is no smiles matching
[INFO] Processed 7500 / 10534 molecules
[INFO] Processed 8000 / 10534 molecules


[00:08:05] Explicit valence for atom # 5 N, 4, is greater than permitted
[00:08:05] Explicit valence for atom # 10 N, 4, is greater than permitted


there is no smiles matching
there is no smiles matching
[INFO] Processed 8500 / 10534 molecules


[00:08:10] Explicit valence for atom # 6 N, 4, is greater than permitted


there is no smiles matching
[INFO] Processed 9000 / 10534 molecules
[INFO] Processed 9500 / 10534 molecules
[INFO] Processed 10000 / 10534 molecules
[INFO] Processed 10500 / 10534 molecules


In [161]:
df.rename(columns={'NAME' : 'Name'}, inplace=True)
print(df.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10534 entries, 0 to 10533
Data columns (total 9 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   ChemID            10534 non-null  int64  
 1   CAS               10534 non-null  object 
 2   SMILES            10534 non-null  object 
 3   Name              10534 non-null  object 
 4   MPID              10534 non-null  object 
 5   Canonical_QSARr   10534 non-null  object 
 6   InChI_Code_QSARr  10534 non-null  object 
 7   InChI Key_QSARr   10534 non-null  object 
 8   LogP              10534 non-null  float64
dtypes: float64(1), int64(1), object(7)
memory usage: 740.8+ KB
None


In [162]:
dups = df["SMILES"].duplicated().sum()
print(f"중복된 SMILES 개수: {dups}")
df = df.drop_duplicates(subset=["SMILES"]).reset_index(drop=True)

중복된 SMILES 개수: 0


In [163]:
desc_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10524 entries, 0 to 10523
Data columns (total 16 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   nHBDon       10524 non-null  int64  
 1   nHBAcc       10524 non-null  int64  
 2   nRing        10524 non-null  int64  
 3   MW           10524 non-null  float64
 4   TopoPSA(NO)  10524 non-null  float64
 5   SLogP        10524 non-null  float64
 6   nAromAtom    10524 non-null  int64  
 7   nRot         10524 non-null  int64  
 8   Kier1        10524 non-null  float64
 9   Kier2        10524 non-null  object 
 10  Kier3        10524 non-null  object 
 11  BalabanJ     10524 non-null  float64
 12  Zagreb1      10524 non-null  float64
 13  BertzCT      10524 non-null  float64
 14  Name         10524 non-null  object 
 15  SMILES       10524 non-null  object 
dtypes: float64(7), int64(5), object(4)
memory usage: 1.3+ MB


In [164]:
print(f"원본 df 개수: {len(df)}")
print(f"유효 SMILES 개수: {sum(df['SMILES'].notna())}")

print(f"desc_df shape: {desc_df.shape}")
print(f"records 개수: {len(records)}")

# 중복된 Canonical_SMILES 있는지 확인
dups = desc_df["Name"].duplicated().sum()
print(f"중복된 Name 개수: {dups}")
desc_df = desc_df.drop_duplicates(subset=["Name", 'SMILES']).reset_index(drop=True)
df = df.drop_duplicates(subset=['Name', 'SMILES']).reset_index(drop=True)



원본 df 개수: 10534
유효 SMILES 개수: 10534
desc_df shape: (10524, 16)
records 개수: 10524
중복된 Name 개수: 227


In [165]:
desc_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10524 entries, 0 to 10523
Data columns (total 16 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   nHBDon       10524 non-null  int64  
 1   nHBAcc       10524 non-null  int64  
 2   nRing        10524 non-null  int64  
 3   MW           10524 non-null  float64
 4   TopoPSA(NO)  10524 non-null  float64
 5   SLogP        10524 non-null  float64
 6   nAromAtom    10524 non-null  int64  
 7   nRot         10524 non-null  int64  
 8   Kier1        10524 non-null  float64
 9   Kier2        10524 non-null  object 
 10  Kier3        10524 non-null  object 
 11  BalabanJ     10524 non-null  float64
 12  Zagreb1      10524 non-null  float64
 13  BertzCT      10524 non-null  float64
 14  Name         10524 non-null  object 
 15  SMILES       10524 non-null  object 
dtypes: float64(7), int64(5), object(4)
memory usage: 1.3+ MB


In [166]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10534 entries, 0 to 10533
Data columns (total 9 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   ChemID            10534 non-null  int64  
 1   CAS               10534 non-null  object 
 2   SMILES            10534 non-null  object 
 3   Name              10534 non-null  object 
 4   MPID              10534 non-null  object 
 5   Canonical_QSARr   10534 non-null  object 
 6   InChI_Code_QSARr  10534 non-null  object 
 7   InChI Key_QSARr   10534 non-null  object 
 8   LogP              10534 non-null  float64
dtypes: float64(1), int64(1), object(7)
memory usage: 740.8+ KB


In [167]:
#merge
df_final = pd.merge(desc_df, df, on="Name", how="outer")

print(f"[DONE] Final shape: {df_final.shape}")

[DONE] Final shape: (12348, 24)


In [168]:
df_final.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12348 entries, 0 to 12347
Data columns (total 24 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   nHBDon            12338 non-null  float64
 1   nHBAcc            12338 non-null  float64
 2   nRing             12338 non-null  float64
 3   MW                12338 non-null  float64
 4   TopoPSA(NO)       12338 non-null  float64
 5   SLogP             12338 non-null  float64
 6   nAromAtom         12338 non-null  float64
 7   nRot              12338 non-null  float64
 8   Kier1             12338 non-null  float64
 9   Kier2             12338 non-null  object 
 10  Kier3             12338 non-null  object 
 11  BalabanJ          12338 non-null  float64
 12  Zagreb1           12338 non-null  float64
 13  BertzCT           12338 non-null  float64
 14  Name              12348 non-null  object 
 15  SMILES_x          12338 non-null  object 
 16  ChemID            12348 non-null  int64 

---

In [169]:
df_missing = df_final[df_final.isna().any(axis=1)]

In [170]:
df_missing

,nHBDon,nHBAcc,nRing,MW,TopoPSA(NO),SLogP,nAromAtom,nRot,Kier1,Kier2,...,Name,SMILES_x,ChemID,CAS,SMILES_y,MPID,Canonical_QSARr,InChI_Code_QSARr,InChI Key_QSARr,LogP
1747,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,"2,4-DiNH2-6-diEtAm-pyrimidine-3-oxide",NaN,112271,NOCAS_879247,[H]C1C(N(C([H])([H])C([H])([H])[H])C([H])([H])...,15884,CCN(CC)C1CC(N)N(=O)C(N)N1,InChI=1S/C8H20N5O/c1-3-12(4-2)7-5-6(9)13(14)8(...,JWBVUUNANBAIIC-UHFFFAOYSA-N,1.16
3503,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,3-METHYL-4-NITROQUINOLINE-1-OXIDE,NaN,105488,14073-00-8,[H]C1C([H])C([H])C2C(C1[H])C([N+](=O)[O-])C(C(...,9868,CC1CN(=O)C2CCCCC2C1[N+]([O-])=O,InChI=1S/C10H17N2O3/c1-7-6-11(13)9-5-3-2-4-8(9...,DGBIVXRSPRWDSZ-UHFFFAOYSA-N,1.06
4438,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,4-CYANOPYRIDINE OXIDE,NaN,105582,14906-59-3,[H]C1C(C#N)C([H])C([H])N(=O)C1[H],9944,N#CC1CCN(=O)CC1,"InChI=1S/C6H9N2O/c7-5-6-1-3-8(9)4-2-6/h6H,1-4H2",MRSXJNPHXOSDAJ-UHFFFAOYSA-N,-0.94
5197,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,"7H-Purine, 6-[[5-[(2-ethoxy-2-oxoethyl)amino]pen",NaN,112230,NOCAS_877618,[H]C1NC2NC([H])N(=O)C(SC([H])([H])C([H])([H])C...,15849,CCOC(=O)CNCCCCCSC1C2NCNC2NCN1=O,InChI=1S/C14H28N5O3S/c1-2-22-11(20)8-15-6-4-3-...,DTFPRFJXWQUHSU-UHFFFAOYSA-N,0.87
6343,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,"BENZOFURAZAN, 1-OXIDE",NaN,101997,480-96-6,[H]C1C([H])C([H])C2C(NON2=O)C1[H],6609,O=N1ONC2CCCCC12,InChI=1S/C6H11N2O2/c9-8-6-4-2-1-3-5(6)7-10-8/h...,AJFSNCPFXDYNDP-UHFFFAOYSA-N,1.43
6570,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,BNEZOCCINNOLINENOXIDE,NaN,113315,6141-98-6,[H]C1C([H])C([H])C2C(NN(=O)C3C([H])C([H])C([H]...,16720,O=N1NC2CCCCC2C2CCCCC12,InChI=1S/C12H21N2O/c15-14-12-8-4-2-6-10(12)9-5...,IUTMFPAQLDGHQY-UHFFFAOYSA-N,2.24
7930,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,"FURAZANAMINE, 4-PHENYL-, 2-OXIDE",NaN,106711,29945-54-8,[H]C1=C([H])C([H])=C(C2NON(=O)C2N([H])[H])C([H...,10972,NC1C(NON1=O)c1ccccc1,InChI=1S/C8H10N3O2/c9-8-7(10-13-11(8)12)6-4-2-...,RYUYTHBIMVPHMG-UHFFFAOYSA-N,1.42
7935,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,"FURAZANCARBOXYLIC ACID, 4-METHYL-, METHYL ESTER,",NaN,110477,104151-78-2,[H]C([H])([H])OC(=O)C1C(C([H])([H])[H])NON1=O,14279,CC1NON(=O)C1C(=O)OC,InChI=1S/C5H9N2O4/c1-3-4(5(8)10-2)7(9)11-6-3/h...,YJSROLATKFZNLP-UHFFFAOYSA-N,0.69
7936,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,"FURAZANCARBOXYLIC ACID, 4-METHYL-, METHYL ESTER,",NaN,110478,104151-90-8,[H]C([H])([H])OC(=O)C1NON(=O)C1C([H])([H])[H],14280,CC1C(NON1=O)C(=O)OC,InChI=1S/C5H9N2O4/c1-3-4(5(8)10-2)6-11-7(3)9/h...,FXWYXKMXQZUWMX-UHFFFAOYSA-N,0.56
10764,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,PYRIDINE-1-OXIDE-4-PHENYL,NaN,103226,1131-61-9,[H]C1=C([H])C([H])=C(C2C([H])C([H])N(=O)C([H])...,7784,O=N1CCC(CC1)c1ccccc1,InChI=1S/C11H14NO/c13-12-8-6-11(7-9-12)10-4-2-...,QFHGVXAIVBLZHD-UHFFFAOYSA-N,0.93


In [171]:
df_weird = df_final[df_final['SMILES_x'] != df_final['SMILES_y']]
len(df_weird)
# 도대체 왜 SMILES가 다르지?

1824

In [172]:
df_weird

,nHBDon,nHBAcc,nRing,MW,TopoPSA(NO),SLogP,nAromAtom,nRot,Kier1,Kier2,...,Name,SMILES_x,ChemID,CAS,SMILES_y,MPID,Canonical_QSARr,InChI_Code_QSARr,InChI Key_QSARr,LogP
140,2.0,7.0,3.0,415.163102,103.32,1.84240,13.0,5.0,24.638672,10.744802,...,"1,2,3-TriMeOPh fused-ring derivative",[H]OC1([H])C([H])([H])C2=C([H])C(OC([H])([H])[...,113933,61036-87-1,[H]C1=C(OC([H])([H])[H])C(=O)C([H])=C2C(=C1[H]...,17267,CCOC(=O)NC1CCc2cc(OC)c(OC)c(OC)c2C2=CC=C(OC)C(...,InChI=1S/C23H27NO7/c1-6-31-23(26)24-16-9-7-13-...,NLHICQCHEWUHCR-UHFFFAOYSA-N,2.30
141,1.0,7.0,3.0,429.178752,92.32,3.48160,13.0,6.0,25.619835,11.92344,...,"1,2,3-TriMeOPh fused-ring derivative",[H]C1=C(OC([H])([H])[H])C(=O)C([H])=C2C(=C1[H]...,113932,61036-87-1,[H]OC1([H])C([H])([H])C2=C([H])C(OC([H])([H])[...,17266,CC(=O)NC1C(O)Cc2cc(OC)c(OC)c(OC)c2C2=CC=C(OC)C...,InChI=1S/C22H25NO7/c1-11(24)23-20-14-10-15(25)...,DBTWJGPROSGGDL-UHFFFAOYSA-N,0.33
157,1.0,6.0,2.0,301.979587,73.80,2.04780,12.0,2.0,14.409972,5.969822,...,"1,2,4-TRIAZIN-5(4H)-ONE, 4-AMINO-6-(3,5-DICHLORO",[H]C1=C(Cl)C([H])=C(C2=NN=C(SC([H])([H])[H])N(...,111817,141627-87-4,[H]C1=C(Cl)C([H])=C(C2=NN=C(N([H])C([H])([H])[...,15472,CNC1=NN=C(C(=O)N1N)c1cc(Cl)cc(Cl)c1,InChI=1S/C10H9Cl2N5O/c1-14-10-16-15-8(9(18)17(...,XASVMTBCGNDVBU-UHFFFAOYSA-N,1.91
158,2.0,6.0,2.0,285.018415,85.83,1.36760,12.0,2.0,14.409972,5.969822,...,"1,2,4-TRIAZIN-5(4H)-ONE, 4-AMINO-6-(3,5-DICHLORO",[H]C1=C(Cl)C([H])=C(C2=NN=C(N([H])C([H])([H])[...,111814,141605-16-5,[H]C1=C(Cl)C([H])=C(C2=NN=C(SC([H])([H])[H])N(...,15470,CSC1=NN=C(C(=O)N1N)c1cc(Cl)cc(Cl)c1,InChI=1S/C10H8Cl2N4OS/c1-18-10-15-14-8(9(17)16...,ATCNTSZOBVZVGV-UHFFFAOYSA-N,3.02
296,2.0,6.0,1.0,249.085973,62.73,2.04830,6.0,5.0,14.062500,6.666667,...,"1,3,5-Triazine,2-difluoromethio-4-i-propylamino-",[H]N(C1=NC(N([H])C([H])(C([H])([H])[H])C([H])(...,112680,103427-39-0,[H]N(C1=NC(SC([H])(F)F)=NC(N([H])C([H])(C([H])...,16236,CC(C)Nc1[n]c(NCC)[n]c([n]1)SC(F)F,InChI=1S/C9H15F2N5S/c1-4-12-7-14-8(13-5(2)3)16...,AKKBSUUVRDJKDW-UHFFFAOYSA-N,3.70
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
11861,1.0,5.0,3.0,349.225308,68.12,4.66710,0.0,5.0,19.753086,8.792244,...,"TETRAHYDROPYRAN-2,4-DIONE,3[1-(ETHOXYIMINO)BUTYL",[H]OC1=C(/C(=N/OC([H])([H])C([H])([H])[H])C([H...,114460,SRC007-42-9,[H]OC1=C(/C(=N/OC([H])([H])C([H])([H])[H])C([H...,17731,CCON=C(CCC)C1=C(O)CC2(CCCCC2C)OC1=O,InChI=1S/C17H27NO4/c1-4-8-13(18-21-5-2)15-14(1...,CMTOTZVHQMSCCH-QGOAFFKASA-N,3.90
11955,1.0,3.0,3.0,284.050715,54.37,3.15040,12.0,2.0,14.917355,6.405827,...,TIOPINAC,[H]OC(=O)C([H])([H])C1=C([H])C([H])=C2SC([H])(...,108306,61220-69-7,[H]OC(=O)C([H])([H])C1=C([H])C([H])=C2C(=O)C3=...,12389,OC(=O)Cc1cc2SCc3ccccc3C(=O)c2cc1,InChI=1S/C16H12O3S/c17-15(18)8-10-5-6-13-14(7-...,KLIVRBFRQSOGQI-UHFFFAOYSA-N,2.97
11956,1.0,3.0,3.0,284.050715,54.37,3.15040,12.0,2.0,14.917355,6.405827,...,TIOPINAC,[H]OC(=O)C([H])([H])C1=C([H])C([H])=C2C(=O)C3=...,107936,61220-69-7,[H]OC(=O)C([H])([H])C1=C([H])C([H])=C2SC([H])(...,12067,OC(=O)Cc1cc2c(cc1)SCc1ccccc1C2=O,InChI=1S/C16H12O3S/c17-15(18)8-10-5-6-14-13(7-...,URRFMRCFYIOQKV-UHFFFAOYSA-N,2.97
12263,0.0,4.0,4.0,295.120843,40.05,3.15130,12.0,2.0,15.523200,7.266436,...,"[1,3]DIOXEPINO[5,6-D]ISOXAZOLE, 3A,4,8,8A-TETRAH",[H]C1=C([H])C([H])=C(C2=NOC3([H])C([H])([H])OC...,111267,123750-56-1,[H]C1=C([H])C([H])=C(C2=NOC3([H])C([H])([H])OC...,14974,Cc1ccccc1C1OCC2ON=C(C2CO1)c1ccccc1,InChI=1S/C19H19NO3/c1-13-7-5-6-10-15(13)19-21-...,RUAPHOBOMXTTFX-UHFFFAOYSA-N,2.72


In [173]:
smi1 = df_weird.loc[140, 'SMILES_x']
smi2 = df_weird.loc[140, 'SMILES_y']

In [174]:
from rdkit import Chem


mol1 = Chem.MolFromSmiles(smi1)
mol2 = Chem.MolFromSmiles(smi2)

print(Chem.MolToSmiles(mol1, canonical=True))
print(Chem.MolToSmiles(mol2, canonical=True))
print(Chem.MolToSmiles(mol1, canonical=True) == Chem.MolToSmiles(mol2, canonical=True))


COc1cc2c(c(OC)c1OC)-c1ccc(OC)c(=O)cc1C(NC(C)=O)C(O)C2
CCOC(=O)NC1CCc2cc(OC)c(OC)c(OC)c2-c2ccc(OC)c(=O)cc21
False


In [175]:
df_final = df_final.drop(index=(df_final[df_final['SMILES_x'] != df_final['SMILES_y']]).index, axis=0)

In [176]:
df_final.info()

<class 'pandas.core.frame.DataFrame'>
Index: 10524 entries, 0 to 12347
Data columns (total 24 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   nHBDon            10524 non-null  float64
 1   nHBAcc            10524 non-null  float64
 2   nRing             10524 non-null  float64
 3   MW                10524 non-null  float64
 4   TopoPSA(NO)       10524 non-null  float64
 5   SLogP             10524 non-null  float64
 6   nAromAtom         10524 non-null  float64
 7   nRot              10524 non-null  float64
 8   Kier1             10524 non-null  float64
 9   Kier2             10524 non-null  object 
 10  Kier3             10524 non-null  object 
 11  BalabanJ          10524 non-null  float64
 12  Zagreb1           10524 non-null  float64
 13  BertzCT           10524 non-null  float64
 14  Name              10524 non-null  object 
 15  SMILES_x          10524 non-null  object 
 16  ChemID            10524 non-null  int64  
 17

In [177]:
df_final.to_csv("train.csv")

---

In [6]:
df = pd.read_csv("cleaned_data/train.csv")

In [ ]:
df.info()
# PFAS를 따는 알고리즘을 가능? -> 넣는 화합물에 맞게 data splitting 해주는 algorithms을 개발?



<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10524 entries, 0 to 10523
Data columns (total 25 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   Unnamed: 0        10524 non-null  int64  
 1   nHBDon            10524 non-null  float64
 2   nHBAcc            10524 non-null  float64
 3   nRing             10524 non-null  float64
 4   MW                10524 non-null  float64
 5   TopoPSA(NO)       10524 non-null  float64
 6   SLogP             10524 non-null  float64
 7   nAromAtom         10524 non-null  float64
 8   nRot              10524 non-null  float64
 9   Kier1             10524 non-null  float64
 10  Kier2             10524 non-null  object 
 11  Kier3             10524 non-null  object 
 12  BalabanJ          10524 non-null  float64
 13  Zagreb1           10524 non-null  float64
 14  BertzCT           10524 non-null  float64
 15  Name              10524 non-null  object 
 16  SMILES_x          10524 non-null  object